# Try the Gemma 4 E2B Japanese Teacher

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peterblazejewicz/gemma-4-e2b-japanese-teacher/blob/main/quick-start.ipynb)

This notebook runs [`blazejewicz/gemma-4-e2b-japanese-teacher-gguf`](https://huggingface.co/blazejewicz/gemma-4-e2b-japanese-teacher-gguf) in
[llama.cpp](https://github.com/ggml-org/llama.cpp) with the system prompts and the sampling of the model card,
and asks it questions as a learner does.

**Before you start:**

1. Open the [model page](https://huggingface.co/blazejewicz/gemma-4-e2b-japanese-teacher-gguf), read the terms, and click **Agree and get access**.
   The files are gated.
2. Make a Hugging Face [access token](https://huggingface.co/settings/tokens) of the type **Read**.
3. In Colab, click the key icon (**Secrets**) on the left, add a secret with the name `HF_TOKEN` and your token as the
   value, and give this notebook access to it. Without the secret, the notebook asks for the token.
4. Select **Runtime > Change runtime type > T4 GPU**. A CPU runtime also works, but each answer then takes tens of
   seconds.
5. Select **Runtime > Run all**.

The GPU build of llama.cpp needs a runtime with Ubuntu 24.04 and an NVIDIA driver 570 or newer. On an older runtime the
notebook uses the CPU, unless you select `COMPILE_FOR_THE_GPU`: then it compiles llama.cpp for the GPU, which takes
20 minutes or more.

The notebook also runs in Jupyter on a Linux x64 computer.

In [ ]:
# @title Settings
DEVICE = "auto"  # @param ["auto", "gpu", "cpu"]
QUANT = "q4_k_m"  # @param ["q4_k_m", "q8_0"]
COMPILE_FOR_THE_GPU = False  # @param {type:"boolean"}
LLAMA_CPP_TAG = "b11223"  # the llama.cpp release that this notebook uses

In [ ]:
# @title 1. Get llama.cpp
%pip install -q huggingface_hub requests

import os
import platform
import shutil
import subprocess
import tarfile
import urllib.request
from pathlib import Path

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise SystemExit("This notebook runs on Linux x64. On another computer, follow the steps of the model card.")

WORK = Path.home() / "gemma-teacher"
WORK.mkdir(exist_ok=True)


def nvidia_driver_major():
    if not shutil.which("nvidia-smi"):
        return None
    out = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    head = out.stdout.strip().split(".")[0]
    return int(head) if head.isdigit() else None


def glibc_version():
    name, version = platform.libc_ver()
    return tuple(int(part) for part in version.split(".")[:2]) if name == "glibc" else (0, 0)


def run_and_show(command, **kwargs):
    """Runs a command and prints its output while it runs."""
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kwargs) as child:
        for line in child.stdout:
            print(line, end="")
    if child.returncode:
        raise SystemExit(f"{command[0]} failed with exit code {child.returncode}")


driver = nvidia_driver_major()
glibc = glibc_version()
if DEVICE == "gpu" and driver is None:
    raise SystemExit("No NVIDIA GPU here. Select Runtime > Change runtime type > T4 GPU, or set DEVICE to cpu.")
use_gpu = driver is not None and DEVICE != "cpu"

# The release build for the CPU needs glibc 2.34 (Ubuntu 22.04). The CUDA 12.8 build needs glibc 2.38
# (Ubuntu 24.04), and it carries only PTX for a T4, which a driver older than 570 cannot compile.
cuda_release_fits = glibc >= (2, 38) and driver is not None and driver >= 570
if use_gpu and not cuda_release_fits and not COMPILE_FOR_THE_GPU:
    print(f"The CUDA build of llama.cpp does not run here (glibc {glibc[0]}.{glibc[1]}, driver {driver}).")
    print("The notebook uses the CPU. For the GPU, use a runtime with Ubuntu 24.04, or select COMPILE_FOR_THE_GPU.")
    use_gpu = False
prebuilt = cuda_release_fits if use_gpu else glibc >= (2, 34)

kind = "ubuntu-cuda-12.8-x64" if use_gpu else "ubuntu-x64"
BIN = WORK / f"llama-{LLAMA_CPP_TAG}-{'gpu' if use_gpu else 'cpu'}"
complete = BIN / ".complete"
if not complete.exists():
    shutil.rmtree(BIN, ignore_errors=True)
    BIN.mkdir()
    if prebuilt:
        base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_CPP_TAG}/"
        names = [f"llama-{LLAMA_CPP_TAG}-bin-{kind}.tar.gz"]
        if use_gpu:
            names.append(f"cudart-llama-{LLAMA_CPP_TAG}-bin-{kind}.tar.gz")
        unpack = WORK / f"unpack-{kind}"
        shutil.rmtree(unpack, ignore_errors=True)
        for name in names:
            print("Downloading", name)
            urllib.request.urlretrieve(base + name, WORK / name)
            with tarfile.open(WORK / name) as tar:
                tar.extractall(unpack, filter="data")
        for folder in unpack.iterdir():
            shutil.copytree(folder, BIN, symlinks=True, dirs_exist_ok=True)
    else:
        print("Compiling llama.cpp for this runtime. On the GPU this takes 20 minutes or more.")
        source = WORK / "llama.cpp"
        if not source.exists():
            run_and_show(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_TAG,
                          "https://github.com/ggml-org/llama.cpp", str(source)])
        flags = ["-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=native"] if use_gpu else []
        env = dict(os.environ)
        if use_gpu and Path("/usr/local/cuda/bin/nvcc").exists():
            env["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
        build = source / f"build-{'gpu' if use_gpu else 'cpu'}"
        run_and_show(["cmake", "-S", str(source), "-B", str(build), "-DCMAKE_BUILD_TYPE=Release", *flags], env=env)
        run_and_show(["cmake", "--build", str(build), "--target", "llama-server",
                      "-j", str(len(os.sched_getaffinity(0)))], env=env)
        shutil.copytree(build / "bin", BIN, symlinks=True, dirs_exist_ok=True)
    complete.touch()

server = BIN / "llama-server"
print(subprocess.run([str(server), "--version"], capture_output=True, text=True).stderr.strip())
print(f"glibc {glibc[0]}.{glibc[1]} | NVIDIA driver {driver}")
print(subprocess.run([str(server), "--list-devices"], capture_output=True, text=True).stdout.strip())

In [ ]:
# @title 2. Download the model (the repository is gated)
from getpass import getpass

from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError


def colab_secret(name):
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


token = (os.environ.get("HF_TOKEN") or colab_secret("HF_TOKEN") or getpass("Hugging Face token: ")).strip()
if not token:
    raise SystemExit("A Hugging Face token of the type Read is necessary. See step 2 above.")
REPO = "blazejewicz/gemma-4-e2b-japanese-teacher-gguf"
try:
    model_path = hf_hub_download(REPO, f"gemma-teacher-e2b-{QUANT}.gguf", token=token)
except GatedRepoError:
    raise SystemExit(f"Open https://huggingface.co/{REPO}, click 'Agree and get access', and run this cell again.")
except HfHubHTTPError as error:
    raise SystemExit(f"The Hub refused the download: {error}. Make sure that the token is correct and can read.")
print(model_path)

In [ ]:
# @title 3. Start llama-server with the settings of the model card
import json
import time
import urllib.error

# The sampling of the model card. llama-server applies it to each request, the chat page too.
SAMPLING_FLAGS = ["--temp", "0.3", "--top-p", "0.95", "--min-p", "0.05",
                  "--repeat-penalty", "1.05", "--repeat-last-n", "64",
                  "--dry-multiplier", "0.8", "--dry-base", "1.75", "--dry-allowed-length", "2",
                  "--dry-penalty-last-n", "128", "--n-predict", "400"]
URL = "http://127.0.0.1:8000"
LOG = WORK / "llama-server.log"

# Stops a llama-server of this notebook that still holds the port, also one of an earlier kernel.
subprocess.run(["pkill", "-f", str(WORK / "llama-")])
time.sleep(1)
proc = subprocess.Popen([str(server), "--model", model_path, "--port", "8000", "--ctx-size", "2048",
                         "--n-gpu-layers", "99" if use_gpu else "0",
                         "--reasoning", "off", "--reasoning-budget", "0", *SAMPLING_FLAGS],
                        stdout=LOG.open("w"), stderr=subprocess.STDOUT)
for _ in range(300):
    if proc.poll() is not None:
        print(LOG.read_text()[-3000:])
        raise SystemExit("llama-server stopped. Its log is above.")
    try:
        with urllib.request.urlopen(URL + "/health", timeout=2) as reply:
            if reply.status == 200 and proc.poll() is None:
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    time.sleep(1)
else:
    raise SystemExit(f"llama-server did not start in 5 minutes; see {LOG}")

# A first request with a long prompt: on a GPU with no kernels of its own in the build (a T4), the
# driver compiles them here, and not during the first question.
started = time.time()
warm_up = json.dumps({"prompt": "日本語を勉強します。" * 60, "n_predict": 8}).encode("utf-8")
urllib.request.urlopen(urllib.request.Request(URL + "/completion", data=warm_up,
                                              headers={"Content-Type": "application/json"}), timeout=600).read()
print("llama-server is ready, on the", "GPU." if use_gpu else "CPU.",
      f"The warm-up request took {time.time() - started:.1f} s.")

In [ ]:
# @title 4. The system prompts of the model card
import json

import requests

BASE_PROMPT = {
    "English": """You are a Japanese language teacher on a small offline device, teaching an
English-speaking beginner (JLPT N5–N4 unless told otherwise).

Rules:
1. Explain in English. Give examples in Japanese, written in normal Japanese
   script (kanji and kana). Never write Japanese in romaji.
2. Keep answers short: at most 2 short paragraphs and at most 2 Japanese
   example sentences. This device SPEAKS your answer aloud and it speaks
   slowly, so every sentence you add is time the learner waits in silence.
3. Do not add furigana, readings, or romaji yourself — the device attaches
   verified readings from a dictionary.
4. If you are not sure about a reading, pitch accent, nuance, or etymology,
   say you are not sure instead of guessing.
5. Stay on Japanese language topics. For anything else, decline in one
   sentence and steer back.
6. In practice mode you will receive the target phrase and the learner's
   transcribed attempt; compare them word by word in English, quoting
   Japanese only in Japanese script.
7. Do not open with praise or an assessment of the learner. No "That is a
   good start", no "Great job", no "Well done". Answer the question or
   correct the sentence, and nothing before that.
8. Do not number your examples and do not announce them. Write the example
   sentence and its English meaning, and no line that says "Here are a few
   more examples".
9. Ask at most one question at the end, and only when it moves the lesson on.""",
    "Polish": """You are a Japanese language teacher on a small offline device, teaching a
Polish-speaking beginner (JLPT N5–N4 unless told otherwise).

Rules:
1. Explain in Polish. Every explanation, comparison and meaning is in
   Polish; never explain in English. Give examples in Japanese, written in
   normal Japanese script (kanji and kana). Never write Japanese in romaji.
2. Keep answers short: at most 2 short paragraphs and at most 2 Japanese
   example sentences. This device SPEAKS your answer aloud and it speaks
   slowly, so every sentence you add is time the learner waits in silence.
3. Do not add furigana, readings, romaji, or a Polish spelling of the sound
   (such as "konniczi wa") yourself — the device attaches
   verified readings from a dictionary.
4. If you are not sure about a reading, pitch accent, nuance, or etymology,
   say in Polish that you are not sure, for example "Nie mam pewności",
   instead of guessing.
5. Polish marks gender in the past tense and in some adjectives. You do not
   know the learner's gender, and the voice that speaks your answer may be
   female or male. Never use a gendered form for yourself or for the
   learner: not "powiedziałem"/"powiedziałam", not "napisałeś"/"napisałaś",
   not "pewien"/"pewna", not "gotowy"/"gotowa". Use the present tense or an
   impersonal form: "Nie mam pewności", "W twojej wersji jest …, a powinno
   być …", "Tu pasuje …". Address the learner as "ty".
6. Stay on Japanese language topics. For anything else, decline in one
   Polish sentence and steer back.
7. In practice mode you will receive the target phrase and the learner's
   transcribed attempt; compare them word by word in Polish, quoting
   Japanese only in Japanese script.
8. Do not open with praise or an assessment of the learner. No "Dobry
   początek", no "Świetne pytanie", no "Świetnie", no "Brawo". Always explain
   the phrase, grammar, or word choice in 1 or 2 Polish sentences before giving
   the Japanese example sentence. Never give a translation alone.
9. Do not number your examples and do not announce them. Put the Japanese
   example sentence on its own line, and its Polish meaning on the next line
   without a dash or bullet points. No line that says "Oto kilka przykładów".
10. Ask at most one question at the end, in Polish, and only when it moves
   the lesson on.
11. The learner's question is in Polish, and your explanation is in Polish even
   when the learner asks how to say something in Japanese. Japanese appears
   only inside 「」 and in the one example sentence.

Answer in the shape of these examples. The learner's words come first, then your answer:

Learner: Jak powiedzieć po japońsku: dobranoc?
You: Po japońsku mówimy 「おやすみなさい」. Mówi się tak przed snem, do rodziny albo do przyjaciół.

おやすみなさい。
Dobranoc.

Learner: Co znaczy mizu?
You: Słowo 「水」 oznacza wodę. To jedno z podstawowych słów w codziennym życiu.

水をください。
Poproszę wodę.

Learner: Popraw moje zdanie: kore wa hon masu.
You: Po rzeczowniku w zdaniu twierdzącym używamy 「です」, a nie ます. Końcówka ます łączy się z czasownikami.

これは本です。
To jest książka.""",
}
LEVEL_FRAGMENT = {
    "beginner": "",
    "intermediate": """The learner is intermediate (JLPT N3) and not a beginner. Write your
Japanese examples at that level.""",
}
LENGTH_FRAGMENT = {
    "brief": """Keep every answer brief: write at most 3 sentences in total, and at most 1
Japanese example sentence.""",
    "normal": "",
    "detailed": """Provide a detailed explanation: explain the grammar structure, nuances, and particle usage in 2 to 3 sentences, followed by 1 or 2 Japanese example sentences.""",
}


def system_prompt(teacher_language, level, length):
    """The system prompt of one combination of the three cards: the base prompt, then the fragments."""
    parts = (BASE_PROMPT[teacher_language], LEVEL_FRAGMENT[level], LENGTH_FRAGMENT[length])
    return "\n\n".join(part for part in parts if part)


def ask(question, teacher_language="English", level="beginner", length="normal"):
    """Streams the teacher's answer to one question and prints the speed of llama-server."""
    body = {"messages": [{"role": "system", "content": system_prompt(teacher_language, level, length)},
                         {"role": "user", "content": question}],
            "stream": True}
    timings = {}
    with requests.post(URL + "/v1/chat/completions", json=body, stream=True, timeout=600) as reply:
        reply.raise_for_status()
        # The stream has no charset, and requests would decode it as Latin-1: decode each line as UTF-8.
        for raw in reply.iter_lines():
            line = raw.decode("utf-8")
            if not line.startswith("data: ") or line == "data: [DONE]":
                continue
            chunk = json.loads(line[len("data: "):])
            timings = chunk.get("timings", timings)
            for choice in chunk.get("choices", []):
                print(choice["delta"].get("content") or "", end="", flush=True)
    if timings:
        cached, new = timings.get("cache_n", 0), timings["prompt_n"]
        print(f"\n\n[prompt {cached + new} tokens: {cached} from the cache of llama-server, {new} new at"
              f" {timings['prompt_per_second']:.0f} tokens/s; answer {timings['predicted_n']} tokens at"
              f" {timings['predicted_per_second']:.1f} tokens/s]")

In [ ]:
# @title 5. Ask the teacher
teacher_language = "Polish"  # @param ["English", "Polish"]
level = "beginner"  # @param ["beginner", "intermediate"]
length = "normal"  # @param ["brief", "normal", "detailed"]
question = "Jak powiedzieć po japońsku: dziękuję za pomoc?"  # @param {type:"string"}
ask(question, teacher_language, level, length)

Change the fields and run the cell again. Some questions to try, from the examples of the model card:

| Teacher language | Level | Length | Question |
| :--- | :--- | :--- | :--- |
| English | beginner | detailed | How do I say, I take the bus to work every morning. |
| English | beginner | normal | What is the difference between は and が? |
| Polish | beginner | brief | Moja córka uczy się grać na pianinie. |
| Polish | intermediate | brief | A jak powiedzieć, mimo że było zimno, poszliśmy nad morze. |

Look at the form of each answer (section 1 of the card): an explanation in the teacher language, an empty line, one
Japanese example sentence, and its translation on the next line. The answers can be wrong; section 8 of the card gives
the limits.